# Fine-tune YOLOv8s on HERIDAL (top-down SAR person detection)
Runtime → Change runtime type → **T4 GPU**. Run cells in order.

HERIDAL: 1,647 drone images (4000×3000) of wilderness, shot straight down from 40–65 m, people labeled. Source: Zenodo record 5662351 (AIR / Aalto Univ., from Božić-Štulić et al. 2019).

In [ ]:
# 1. Setup: GPU check, Drive, Ultralytics
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
# 2. Download HERIDAL (8.3 GB, ~5-10 min) and unzip to Colab disk (not Drive)
!wget -q --show-progress -O /content/heridal.zip "https://zenodo.org/records/5662351/files/heridal_keras_retinanet_voc.zip?download=1"
!unzip -q /content/heridal.zip -d /content/heridal_raw && rm /content/heridal.zip
!find /content/heridal_raw -maxdepth 3 -type d

In [ ]:
# 3. Convert VOC XML -> YOLO and cut images into 640px tiles at full resolution
#    (people are ~30-60 px; shrinking 4000px images to 640 would erase them)
import random, xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
random.seed(0)

RAW, OUT = Path('/content/heridal_raw'), Path('/content/heridal_yolo')
TILE, STRIDE = 640, 480   # 160 px overlap
NEG_KEEP = 0.1            # keep 10% of empty tiles so the model learns 'no person'

ok = lambda p: not p.name.startswith('._') and '__MACOSX' not in str(p)  # skip macOS junk files
imgs = {p.stem: p for p in RAW.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and ok(p)}
xmls = [x for x in RAW.rglob('*.xml') if ok(x)]
print(len(imgs), 'images,', len(xmls), 'xml files')

def read_boxes(x):
    out = []
    for o in ET.parse(x).getroot().iter('object'):
        b = o.find('bndbox')
        out.append(tuple(float(b.find(k).text) for k in ('xmin', 'ymin', 'xmax', 'ymax')))
    return out

items = [(imgs[x.stem], read_boxes(x), 'test' if 'test' in str(x).lower() else 'train')
         for x in xmls if x.stem in imgs]
train = [i for i in items if i[2] == 'train']
test = [i for i in items if i[2] == 'test']
random.shuffle(train)
nval = len(train) // 10
splits = {'val': train[:nval], 'train': train[nval:], 'test': test}
print({k: len(v) for k, v in splits.items()}, 'full images')

def starts(L):
    s = list(range(0, max(L - TILE, 0) + 1, STRIDE))
    if s[-1] + TILE < L:
        s.append(L - TILE)
    return s

for split, its in splits.items():
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)
    n_tiles = n_people = 0
    for path, bxs, _ in its:
        im = Image.open(path).convert('RGB')
        W, H = im.size
        for y0 in starts(H):
            for x0 in starts(W):
                lab = []
                for a, b, c, d in bxs:
                    ca, cb = max(a, x0), max(b, y0)
                    cc, cd = min(c, x0 + TILE), min(d, y0 + TILE)
                    if cc <= ca or cd <= cb:
                        continue
                    if (cc - ca) * (cd - cb) < 0.5 * (c - a) * (d - b):
                        continue  # person mostly outside this tile
                    lab.append(f'0 {((ca+cc)/2-x0)/TILE:.6f} {((cb+cd)/2-y0)/TILE:.6f} '
                               f'{(cc-ca)/TILE:.6f} {(cd-cb)/TILE:.6f}')
                if not lab and random.random() > NEG_KEEP:
                    continue
                name = f'{path.stem}_{x0}_{y0}'
                im.crop((x0, y0, x0 + TILE, y0 + TILE)).save(OUT / split / 'images' / f'{name}.jpg', quality=95)
                (OUT / split / 'labels' / f'{name}.txt').write_text('\n'.join(lab))
                n_tiles += 1
                n_people += len(lab)
    print(f'{split}: {len(its)} images -> {n_tiles} tiles, {n_people} person boxes')

(OUT / 'data.yaml').write_text(
    f'path: {OUT}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n  0: person\n')

In [ ]:
# 4. Train (~1.5-3 h on T4). Results + weights save to Drive: sar_runs/heridal_s
from ultralytics import YOLO
model = YOLO('yolov8s.pt')  # start from COCO person knowledge
model.train(data='/content/heridal_yolo/data.yaml', imgsz=640, epochs=40, batch=16,
            patience=10, project='/content/drive/MyDrive/sar_runs', name='heridal_s')

In [ ]:
# 4b. ONLY if Colab disconnected mid-training: rerun cells 1-3, then this instead of 4
from ultralytics import YOLO
YOLO('/content/drive/MyDrive/sar_runs/heridal_s/weights/last.pt').train(resume=True)

In [ ]:
# 5. Score on the held-out HERIDAL test set (never seen in training)
from ultralytics import YOLO
m = YOLO('/content/drive/MyDrive/sar_runs/heridal_s/weights/best.pt')
r = m.val(data='/content/heridal_yolo/data.yaml', split='test', imgsz=640)
print(f'Test  precision {r.box.mp:.3f} | recall {r.box.mr:.3f} | mAP50 {r.box.map50:.3f}')